In [ ]:
from pathlib import Path
import sys

import geopandas as gpd
import pandas as pd
from IPython.display import display

root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src/analysis.py").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Open this notebook from the project or notebooks directory.")
sys.path.insert(0, str(root))
from src.analysis import PICKUP_COLUMNS, pickup_sources

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:.5f}")

In [ ]:
frames = []
quality = []
for month, path in pickup_sources().items():
    frame = pd.read_csv(path, usecols=PICKUP_COLUMNS)
    quality.append({
        "month": month,
        "records": len(frame),
        "missing_cells": int(frame.isna().sum().sum()),
        "repeated_rows_retained": int(frame.duplicated().sum()),
    })
    frames.append(frame)

df_raw = pd.concat(frames, ignore_index=True)
gdf_uber = gpd.GeoDataFrame(
    df_raw,
    geometry=gpd.points_from_xy(df_raw["Lon"], df_raw["Lat"]),
    crs="EPSG:4326",
)
display(pd.DataFrame(quality), gdf_uber.head())